In [29]:
import pandas as pd
ratings = pd.read_csv(
    "./ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)
ratings = ratings[["userId", "movieId", "timestamp", "rating"]]
ratings.head()



,userId,movieId,timestamp,rating
0,1,1193,978300760,5
1,1,661,978302109,3
2,1,914,978301968,3
3,1,3408,978300275,4
4,1,2355,978824291,5


In [30]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user2idx = {u:i for i, u in enumerate(user_ids)}
movie2idx = {m:i for i, m in enumerate(movie_ids)}

ratings["user"] = ratings["userId"].map(user2idx)
ratings["movie"] = ratings["movieId"].map(movie2idx)

ratings["rating"] = ratings["rating"].astype("float32")
ratings["rating"] = (ratings["rating"] - ratings["rating"].mean()) / ratings["rating"].std()

In [31]:
ratings

,userId,movieId,timestamp,rating,user,movie
0,1,1193,978300760,1.269746,0,0
1,1,661,978302109,-0.520601,0,1
2,1,914,978301968,-0.520601,0,2
3,1,3408,978300275,0.374572,0,3
4,1,2355,978824291,1.269746,0,4
...,...,...,...,...,...,...
1000204,6040,1091,956716541,-2.310948,6039,772
1000205,6040,1094,956704887,1.269746,6039,1106
1000206,6040,562,956704746,1.269746,6039,365
1000207,6040,1096,956715648,0.374572,6039,152


In [32]:
num_users = ratings["user"].nunique()
num_movies = ratings["movie"].nunique()

print(num_users, num_movies)

6040 3706


In [33]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size = 0.2, random_state = 42)

In [34]:
import torch
import torch.nn as nn
train_user = torch.tensor(train["user"].values)
train_movie = torch.tensor(train["movie"].values)
train_rating = torch.tensor(train["rating"].values, dtype = torch.float32)

In [ ]:
class Recommender(nn.Module):

    def __init__(self, num_users, num_movies, emb_size=50):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, emb_size)
        self.movie_emb = nn.Embedding(num_movies, emb_size)

        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)

    def forward(self, user, movie):

        u = self.user_emb(user)
        m = self.movie_emb(movie)

        dot = (u * m).sum(1)

        bias = self.user_bias(user).squeeze() + self.movie_bias(movie).squeeze()

        return dot + bias

In [ ]:
model = Recommender(num_users, num_movies)

In [44]:
loss_fn = nn.MSELoss()
initial_lambda = 1e-4
for epoch in range(100):

    current_lambda = initial_lambda * (0.75 ** epoch)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=current_lambda
    )
    pred = model(train_user, train_movie)

    loss = loss_fn(pred, train_rating)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(epoch, loss.item())

0 0.5955088138580322
1 0.5954316258430481
2 0.5944004058837891
3 0.592742383480072
4 0.5907750129699707
5 0.5886710286140442
6 0.5865079760551453
7 0.5843185782432556
8 0.582115650177002
9 0.5799053907394409
10 0.5776907205581665
11 0.5754730105400085
12 0.5732529759407043
13 0.5710312128067017
14 0.5688079595565796
15 0.5665834546089172
16 0.5643579363822937
17 0.5621315836906433
18 0.5599046945571899
19 0.5576772689819336
20 0.5554496645927429
21 0.5532218813896179
22 0.5509939789772034
23 0.5487661361694336
24 0.546538233757019
25 0.5443108081817627
26 0.5420833826065063
27 0.5398563146591187
28 0.5376295447349548
29 0.5354032516479492
30 0.5331773161888123
31 0.5309517979621887
32 0.5287266373634338
33 0.5265018343925476
34 0.52427738904953
35 0.5220533013343811
36 0.5198296904563904
37 0.5176064968109131
38 0.5153836011886597
39 0.5131612420082092
40 0.5109393000602722
41 0.5087178349494934
42 0.5064966678619385
43 0.5042760372161865
44 0.5020560026168823
45 0.49983635544776917
46

Instead use batches

In [38]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(train_user, train_movie, train_rating)

loader = DataLoader(dataset, batch_size=4096, shuffle=True)

In [43]:
initial_lambda = 1e-4

for epoch in range(20):

    total_loss = 0
    current_lambda = initial_lambda * (0.95** epoch)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=current_lambda
    )
    for u,m,r in loader:

        pred = model(u,m)

        loss = loss_fn(pred,r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(epoch, total_loss)

KeyboardInterrupt: 

In [41]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.002, weight_decay=initial_lambda)

for epoch in range(20):
    
    for g in optimizer.param_groups:
        g['weight_decay'] = initial_lambda * (0.95 ** epoch)

    total_loss = 0
    for u,m,r in loader:
        pred = model(u,m)
        loss = loss_fn(pred,r)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(epoch, total_loss)

0 3079.5192365646362
1 2087.4634981155396
2 1459.0481204986572
3 1045.2315421104431
4 766.9951460361481
5 575.4992063045502
6 441.8931666612625
7 347.3325927257538
8 279.6431403160095
9 230.49068558216095
10 194.73140758275986
11 168.3099417090416
12 148.78599858283997
13 134.1999650001526
14 123.25221359729767
15 115.00346213579178
16 108.73271244764328
17 103.90233325958252
18 100.12257134914398
19 97.08961701393127
